In [1]:
import pandas as pd
import numpy as np

# 1. Setup Time Index for a standard non-leap year (8760 hours)
time_index = pd.date_range(start="2024-01-01 00:00:00", periods=8760, freq="h")
df = pd.DataFrame(index=time_index)
df['Hour_of_Year'] = np.arange(1, 8761)
df['Hour_of_Day'] = df.index.hour
df['Month'] = df.index.month

# 2. Generate Synthetic Coastal Weather (SI Units)
# Oregon coast: Mean ~11.1 C, Winter lows ~2 C, Summer highs ~21 C
base_temp_c = 11.1
annual_temp_variation_c = -6.7 * np.cos(2 * np.pi * (df.index.dayofyear - 15) / 365)
diurnal_temp_variation_c = -4.4 * np.cos(2 * np.pi * (df['Hour_of_Day'] - 4) / 24)

df['Outdoor_Temp_C'] = base_temp_c + annual_temp_variation_c + diurnal_temp_variation_c
np.random.seed(42) # For reproducibility
df['Outdoor_Temp_C'] += np.random.normal(0, 1.5, 8760) # Gaussian weather noise
df['Outdoor_Temp_C'] = df['Outdoor_Temp_C'].round(1)

# 3. Generate Hydrological Profile (Streamflow in m^3/s)
# Oregon coastal streams peak in Jan/Feb and hit baseflow in Aug/Sept
base_flow_m3s = 0.12 
annual_flow_variation = 0.07 * np.cos(2 * np.pi * (df.index.dayofyear - 15) / 365)
# Add lognormal noise to simulate sharp spikes from Pacific rainstorms
rain_spikes = np.random.lognormal(mean=-3, sigma=1.2, size=8760) * 0.05

df['Streamflow_m3_s'] = base_flow_m3s + annual_flow_variation + rain_spikes
# Floor the streamflow to the absolute minimum 1.8 cfs (0.051 m^3/s) specified in the brief
df['Streamflow_m3_s'] = df['Streamflow_m3_s'].clip(lower=0.051).round(3)

# 4. Calculate Thermal Loads (SI Units)
# UA Value: 300 BTU/hr-F equates to ~158 W/C. Rounded to 0.16 kW/C for clean math.
# UA_VALUE_kW_C = 0.16
# T_SET_HEAT_C = 20.0
# T_SET_COOL_C = 24.0

# df['Heating_Load_kWh'] = np.maximum(0, (T_SET_HEAT_C - df['Outdoor_Temp_C']) * UA_VALUE_kW_C).round(2)
# df['Cooling_Load_kWh'] = np.maximum(0, (df['Outdoor_Temp_C'] - T_SET_COOL_C) * UA_VALUE_kW_C).round(2)

# 5. Generate Disaggregated Electrical Loads (kW)
# A. Base Load (Continuous draw from fridge, router, standby devices)
df['Base_Load_kW'] = 0.35 + np.random.uniform(0, 0.05, 8760)

# B. Lighting & Plug Loads (Morning and evening peaks)
morning_lp = 0.4 * np.exp(-0.5 * ((df['Hour_of_Day'] - 7) / 1.5)**2)
evening_lp = 0.6 * np.exp(-0.5 * ((df['Hour_of_Day'] - 19) / 2.0)**2)
df['Lighting_Plugs_kW'] = morning_lp + evening_lp + np.random.uniform(0, 0.10, 8760)

# C. Induction Range (Cooking energy)
df['Induction_Range_kW'] = 0.0
breakfast_mask = (df['Hour_of_Day'] >= 6) & (df['Hour_of_Day'] <= 8)
dinner_mask = (df['Hour_of_Day'] >= 17) & (df['Hour_of_Day'] <= 19)
df.loc[breakfast_mask, 'Induction_Range_kW'] = np.random.uniform(0.2, 1.2, size=breakfast_mask.sum())
df.loc[dinner_mask, 'Induction_Range_kW'] = np.random.uniform(0.5, 2.5, size=dinner_mask.sum())

# D. Total Hourly Load
df['Total_Electrical_Load_kW'] = df['Base_Load_kW'] + df['Lighting_Plugs_kW'] + df['Induction_Range_kW']

# 6. Generate Domestic Hot Water (DHW) Demand (Liters per hour)
df['DHW_Demand_Liters'] = 0.0
# Morning shower volume (up to ~55 Liters / 15 gal)
df.loc[breakfast_mask, 'DHW_Demand_Liters'] = np.random.uniform(0, 55, size=breakfast_mask.sum())
# Evening washing/dishes volume (up to ~30 Liters / 8 gal)
df.loc[dinner_mask, 'DHW_Demand_Liters'] = np.random.uniform(0, 30, size=dinner_mask.sum())

# 7. Clean up rounding for all columns
df['Base_Load_kW'] = df['Base_Load_kW'].round(2)
df['Lighting_Plugs_kW'] = df['Lighting_Plugs_kW'].round(2)
df['Induction_Range_kW'] = df['Induction_Range_kW'].round(2)
df['Total_Electrical_Load_kW'] = df['Total_Electrical_Load_kW'].round(2)
df['DHW_Demand_Liters'] = df['DHW_Demand_Liters'].round(1)

# 8. Export to CSV
columns_to_export = [
    'Hour_of_Year', 'Hour_of_Day', 'Outdoor_Temp_C', 'Streamflow_m3_s',
    'Base_Load_kW', 'Lighting_Plugs_kW', 'Induction_Range_kW', 'Total_Electrical_Load_kW',
    'DHW_Demand_Liters', 'Heating_Load_kWh', 'Cooling_Load_kWh'
]
columns_to_export = [
    'Hour_of_Year', 'Hour_of_Day', 'Outdoor_Temp_C', 'Streamflow_m3_s',
    'Base_Load_kW', 'Lighting_Plugs_kW', 'Induction_Range_kW', 'Total_Electrical_Load_kW',
    'DHW_Demand_Liters'
]
output_df = df[columns_to_export]

file_name = 'Yachats_Off_Grid_2024.csv'
output_df.to_csv(file_name, index=False)
print(f"Successfully generated {file_name} with disaggregated SI data.")

Successfully generated Yachats_Off_Grid_2024.csv with disaggregated SI data.


In [3]:
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

plot_columns = [
    'Outdoor_Temp_C', 'Streamflow_m3_s',
    'Base_Load_kW', 'Lighting_Plugs_kW',
    'Induction_Range_kW', 'Total_Electrical_Load_kW',
    'DHW_Demand_Liters'
]

df_plot = output_df.copy()
df_plot['Timestamp'] = pd.date_range(start="2024-01-01 00:00:00", periods=len(df_plot), freq="h")
df_plot.set_index('Timestamp', inplace=True)

range_selector = [
    dict(count=1, label="1d", step="day", stepmode="backward"),
    dict(count=7, label="7d", step="day", stepmode="backward"),
    dict(count=1, label="1m", step="month", stepmode="backward"),
    dict(count=3, label="3m", step="month", stepmode="backward"),
    dict(step="all")
]

def update_plot(variable, start_date, end_date, daily_mean):
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)
    if start_date > end_date:
        start_date, end_date = end_date, start_date

    data = df_plot.loc[start_date:end_date, [variable]]
    if daily_mean:
        data = data.resample("D").mean()
        mode_label = "Daily average"
    else:
        mode_label = "Hourly"

    data = data.reset_index()
    fig = px.line(
        data,
        x='Timestamp',
        y=variable,
        title=f"{variable} ({mode_label})",
        labels={variable: variable, "Timestamp": "Date"}
    )
    fig.update_layout(
        xaxis=dict(
            rangeslider=dict(visible=True),
            rangeselector=dict(buttons=range_selector)
        ),
        margin=dict(l=40, r=20, t=50, b=30)
    )
    fig.show()

variable_picker = widgets.Dropdown(
    options=plot_columns,
    value='Total_Electrical_Load_kW',
    description="Metric:"
)

start_picker = widgets.DatePicker(
    value=pd.to_datetime("2024-01-01"),
    description="Start"
)

end_picker = widgets.DatePicker(
    value=pd.to_datetime("2024-12-31"),
    description="End"
)

daily_checkbox = widgets.Checkbox(
    value=False,
    description="Daily average"
)

controls = widgets.VBox([
    widgets.HBox([variable_picker, daily_checkbox]),
    widgets.HBox([start_picker, end_picker])
])

output = widgets.interactive_output(
    update_plot,
    {
        "variable": variable_picker,
        "start_date": start_picker,
        "end_date": end_picker,
        "daily_mean": daily_checkbox
    }
)

display(controls, output)

Output()